#### Objectif

Ce notebook sert de workflow d'orchestration pour la partie exploratoire Morlet. 
Toute la logique de lecture, prétraitement, epoching, calcul temps-fréquence, normalisation baseline et génération de figures est centralisée dans lfp_morlet_utils.py.

Structure de travail : 
root_dir/
    └── results_morlet_exploratoire/
        └── PATIENT_stimN/                                               {créé}
            ├── PATIENT_stimN_baseline_values_<contact_pair>.npy
            ├── PATIENT_stimN_power_pre<contact_pair>.npy
            ├── PATIENT_stimN_power_post<contact_pair>.npy
            ├── PATIENT_stimN_logratio_<contact_pair>.npy
            ├── PATIENT_stimN_percent<contact_pair>.npy
            ├── PATIENT_stimN_subtract<contact_pair>.npy
            ├── PATIENT_stimN_zscore<contact_pair>.npy
            ├── PATIENT_stimN_freqs.npy
            ├── PATIENT_stimN_pre_times.npy
            ├── PATIENT_stimN_post_times.npy
            ├── PATIENT_stimN_stims_table.csv
            └── PATIENT_stimN_metadata.json
        ...
        └── run_summary.json                                              {créé}

    ├── PATIENT_stimN.TRC                                                 {nécessaire}
    ├── PATIENT_stimN_stim_events_TRC.txt                                 {nécessaire}
    ├── PATIENT_stimN_stim_sevents_TRC_shifted.txt                        {nécessaire}
    ├── PATIENT_stimN_stim_events_TRC_re-shifted.txt                      {nécessaire}
    ├── PATIENT_stimN_stim_events_TRC_re-shifted_loca_COG.txt             {nécessaire}
    ...
    └── TRC_bad_channels.xlsx                                             {nécessaire}

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from utils_time_frequency.lfp_preprocess_utils import (
    list_trc_sessions,
    load_bad_channels_table
)

from utils_time_frequency.lfp_morlet_utils import (
    MorletConfig,
    prepare_session_data,
    run_session_morlet,
    run_all_sessions_morlet,
    load_session_exports,
    generate_session_figures_from_exports
)

In [3]:
# configuration Morlet

morlet_cfg = MorletConfig(
    root_dir="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog",
    output_dir="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_morlet_exploratoire",

    # Epoching
    pre_length=3.0,
    post_length=3.0,
    epsilon=0.1,
    allow_global_baseline=True,

    # Filtrage
    do_notch=True,
    notch_freqs=(50.0, 100.0, 150.0),
    notch_q=30.0,
    do_highpass=True,
    highpass_hz=0.1,

    # Morlet
    fmin=4.0,
    fmax=150.0,
    n_freqs=20,
    freq_scale="linear",
    n_cycles=7.0,
    decim=16,
    n_jobs=1,

    # Baseline
    baseline_mode="trial_pre",          # recommandé pour la phase exploratoire
    baseline_stat="median",
    metrics_to_compute=("logratio", "percent", "zscore", "subtract"),
    eps=1e-12,

    # Figures
    make_figures=True,
    figure_dpi=150,
    max_cols_per_figure=3,
    cmap_raw="viridis",
    cmap_metric_div="RdBu_r",
    raw_display_mode="log10",
    z_threshold=3.0,
    lineplot_time_stat="mean",
    save_png=True,

    # Logs
    verbose=True,
)

In [4]:
# quelles sessions dispo ?
root_dir = Path(morlet_cfg.root_dir)
sessions = list_trc_sessions(root_dir)
print(f"{len(sessions)} sessions trouvées")
sessions

1 sessions trouvées


['P119_FM71_stim4']

In [5]:
# quels bad channels ?
bad_df = load_bad_channels_table(root_dir)
bad_df

,session,bad_channels
0,P64_BR34_stim2,"B14,C15,GC6,GC7,W46"
1,P97_BM50_stim1,GC'4
2,P101_DC54_stim2,"B_1,H_11,CU_12,Bp8"
3,P101_DC54_stim4,B_1
4,P119_FM71_stim4,"Bp1,Bp2,Bp3,Bp6,Bp7,Bp8"


#### Morlet / une session

In [6]:
# pour test sur session unique
session = sessions[0]   # ou nom spécifique de session
session

'P119_FM71_stim4'

In [8]:
# preparation de la session (a mettre dans boucle sur sessions apres)

session_data = prepare_session_data(
    session=session,
    root_dir=root_dir,
    bad_df=bad_df,
    cfg=morlet_cfg,
)

print("Clés disponibles :", session_data.keys())
print("Session :", session_data["session"])
print("sfreq :", session_data["sfreq"])
print("n_stims :", len(session_data["stims_df"]))
print("n_raw_channels :", len(session_data["raw_ch_names"]))
print("n_bad_channels :", len(session_data["bad_channels"]))
print("n_bipolar_channels :", len(session_data["bp_names"]))
print("pre_epochs shape :", session_data["pre_epochs"].shape)
print("post_epochs shape :", session_data["post_epochs"].shape)

session_data


=== Préparation session P119_FM71_stim4 ===
391.859
<class 'numpy.float64'>
[INFO] P119_FM71_stim4.TRC: sfreq = 512.0
[INFO] P119_FM71_stim4.TRC: unités canal détectées = ['%', 'bpm', 'dimentionless', 'μV']
Clés disponibles : dict_keys(['session', 'sfreq', 'stims_df', 'raw_ch_names', 'bad_channels', 'bipolar_pairs', 'bp_names', 'data_bp', 'pre_epochs', 'post_epochs', 'pre_times', 'post_times'])
Session : P119_FM71_stim4
sfreq : 512.0
n_stims : 31
n_raw_channels : 136
n_bad_channels : 6
n_bipolar_channels : 85
pre_epochs shape : (31, 85, 1536)
post_epochs shape : (31, 85, 1536)


{'session': 'P119_FM71_stim4',
 'sfreq': 512.0,
 'stims_df':                       label_stim   t_start  duration         lobe  \
 0      Tp3-Tp42.0mA7.0Hz1025µsec   307.656   9.86499     L Insula   
 1      Tp5-Tp62.0mA7.0Hz1025µsec   357.097   9.87872   L Temporal   
 2      Tp8-Tp92.0mA7.0Hz1025µsec   391.859   9.88953   L Temporal   
 3     Hp9-Hp102.0mA7.0Hz1025µsec   445.130   9.86789   L Temporal   
 4      Hp5-Hp62.0mA7.0Hz1025µsec   474.216   9.86789   L Temporal   
 5      Hp4-Hp52.0mA7.0Hz1025µsec   528.119   9.87872   L Temporal   
 6      Hp1-Hp22.0mA7.0Hz1025µsec   604.054   9.88953   L Temporal   
 7      Bp6-Bp72.0mA7.0Hz1025µsec   670.108   9.88953   L Temporal   
 8     CP1-CP21.2mA50.0Hz1025µsec   927.946   4.99887  R Cingulate   
 9       X7-X81.2mA50.0Hz1025µsec   996.349   4.98804     R Insula   
 10      X5-X61.2mA50.0Hz1025µsec  1021.979   4.95560     R Insula   
 11      X3-X41.2mA50.0Hz1025µsec  1046.349   4.93393     R Insula   
 12      X3-X41.2mA50.0Hz1025µ

In [9]:
# Morlet sur cette session 
session_out = run_session_morlet(
    session_data=session_data,
    out_dir=Path(morlet_cfg.output_dir),
    cfg=morlet_cfg,
)

session_out


=== Morlet session P119_FM71_stim4 ===
[INFO] P119_FM71_stim4: Morlet canal 1/85 -> TPp1-TPp2
[INFO] P119_FM71_stim4 TPp1-TPp2: pre_ep_ch shape = (31, 1, 1536)
[INFO] P119_FM71_stim4 TPp1-TPp2: post_ep_ch shape = (31, 1, 1536)
[INFO] P119_FM71_stim4 TPp1-TPp2: power_pre shape = (31, 1, 20, 96)
[INFO] P119_FM71_stim4 TPp1-TPp2: power_post shape = (31, 1, 20, 96)
[INFO] P119_FM71_stim4 TPp1-TPp2: sauvegarde canal OK
[INFO] P119_FM71_stim4: Morlet canal 2/85 -> TPp2-TPp3
[INFO] P119_FM71_stim4 TPp2-TPp3: pre_ep_ch shape = (31, 1, 1536)
[INFO] P119_FM71_stim4 TPp2-TPp3: post_ep_ch shape = (31, 1, 1536)
[INFO] P119_FM71_stim4 TPp2-TPp3: power_pre shape = (31, 1, 20, 96)
[INFO] P119_FM71_stim4 TPp2-TPp3: power_post shape = (31, 1, 20, 96)
[INFO] P119_FM71_stim4 TPp2-TPp3: sauvegarde canal OK
[INFO] P119_FM71_stim4: Morlet canal 3/85 -> TPp5-TPp6
[INFO] P119_FM71_stim4 TPp5-TPp6: pre_ep_ch shape = (31, 1, 1536)
[INFO] P119_FM71_stim4 TPp5-TPp6: post_ep_ch shape = (31, 1, 1536)
[INFO] P119_FM

PosixPath('/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_morlet_exploratoire/P119_FM71_stim4')

In [10]:
# reload exports de Morlet pour verification
session_out = Path(morlet_cfg.output_dir + '/' + session)
exports = load_session_exports(session_out, session)

print("Session exportée :", exports["session"])
print("freqs shape :", exports["freqs"].shape)
print("post_times shape :", exports["post_times"].shape)
print("n_trials exportés :", len(exports["stims_df"]))
print("n_bipolar_channels exportés :", len(exports["metadata"]["bipolar_names"]))

Session exportée : P119_FM71_stim4
freqs shape : (20,)
post_times shape : (96,)
n_trials exportés : 31
n_bipolar_channels exportés : 85


In [ ]:
# regenerate figures from exports (from .npy)    (30 min pour une session à 30 stims)
session_out = Path(morlet_cfg.output_dir + '/' + session)
generate_session_figures_from_exports(
    session_dir=session_out,
    cfg=morlet_cfg,
)

[OK] P119_FM71_stim4: figures exploratoires générées


#### Morlet / toutes les sessions

In [17]:
# Morlet sur toutes les sessions
summary_morlet = run_all_sessions_morlet(morlet_cfg)
summary_morlet

2 sessions TRC trouvées

=== Préparation session P101_DC54_stim2 ===


[INFO] P101_DC54_stim2.TRC: sfreq = 2048.0
[INFO] P101_DC54_stim2.TRC: unités canal détectées = ['%', 'bpm', 'dimentionless', 'μV']

=== Morlet session P101_DC54_stim2 ===
[INFO] P101_DC54_stim2: Morlet canal 1/89 -> A1-A2
[INFO] P101_DC54_stim2 A1-A2: pre_ep_ch shape = (42, 1, 6144)
[INFO] P101_DC54_stim2 A1-A2: post_ep_ch shape = (42, 1, 6144)
[INFO] P101_DC54_stim2 A1-A2: power_pre shape = (42, 1, 20, 384)
[INFO] P101_DC54_stim2 A1-A2: power_post shape = (42, 1, 20, 384)
[INFO] P101_DC54_stim2 A1-A2: sauvegarde canal OK
[INFO] P101_DC54_stim2: Morlet canal 2/89 -> A2-A3
[INFO] P101_DC54_stim2 A2-A3: pre_ep_ch shape = (42, 1, 6144)
[INFO] P101_DC54_stim2 A2-A3: post_ep_ch shape = (42, 1, 6144)
[INFO] P101_DC54_stim2 A2-A3: power_pre shape = (42, 1, 20, 384)
[INFO] P101_DC54_stim2 A2-A3: power_post shape = (42, 1, 20, 384)
[INFO] P101_DC54_stim2 A2-A3: sauvegarde canal OK
[INFO] P101_DC54_stim2: Morlet canal 3/89 -> A6-A7
[INFO] P101_DC54_stim2 A6-A7: pre_ep_ch shape = (42, 1, 6144)
[

[INFO] P64_BR34_stim2.TRC: sfreq = 2048.0
[INFO] P64_BR34_stim2.TRC: unités canal détectées = ['%', 'bpm', 'dimentionless', 'μV']

=== Morlet session P64_BR34_stim2 ===
[INFO] P64_BR34_stim2: Morlet canal 1/110 -> A1-A2
[INFO] P64_BR34_stim2 A1-A2: pre_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34_stim2 A1-A2: post_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34_stim2 A1-A2: power_pre shape = (15, 1, 20, 384)
[INFO] P64_BR34_stim2 A1-A2: power_post shape = (15, 1, 20, 384)
[INFO] P64_BR34_stim2 A1-A2: sauvegarde canal OK
[INFO] P64_BR34_stim2: Morlet canal 2/110 -> A2-A3
[INFO] P64_BR34_stim2 A2-A3: pre_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34_stim2 A2-A3: post_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34_stim2 A2-A3: power_pre shape = (15, 1, 20, 384)
[INFO] P64_BR34_stim2 A2-A3: power_post shape = (15, 1, 20, 384)
[INFO] P64_BR34_stim2 A2-A3: sauvegarde canal OK
[INFO] P64_BR34_stim2: Morlet canal 3/110 -> A3-A4
[INFO] P64_BR34_stim2 A3-A4: pre_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34

{'n_sessions': 2,
 'n_errors': 0,
 'errors': [],
 'config': {'root_dir': '/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog',
  'output_dir': '/home/aube/Documents/article_neuronal_stimic/effets_cog//results_morlet_exploratoire',
  'pre_length': 3.0,
  'post_length': 3.0,
  'epsilon': 0.1,
  'allow_global_baseline': True,
  'do_notch': True,
  'notch_freqs': (50.0, 100.0, 150.0),
  'notch_q': 30.0,
  'do_highpass': True,
  'highpass_hz': 0.1,
  'fmin': 4.0,
  'fmax': 150.0,
  'n_freqs': 20,
  'freq_scale': 'linear',
  'n_cycles': 7.0,
  'decim': 16,
  'n_jobs': 1,
  'baseline_mode': 'trial_pre',
  'baseline_stat': 'median',
  'metrics_to_compute': ('logratio', 'percent', 'zscore', 'subtract'),
  'eps': 1e-12,
  'save_raw_epochs': False,
  'save_filtered_signal_preview': False,
  'make_figures': True,
  'figure_dpi': 150,
  'max_cols_per_figure': 3,
  'cmap_raw': 'viridis',
  'cmap_metric_div': 'RdBu_r',
  'raw_display_mode': 'log10',
  'z_threshold': 3.0,
  'linep

In [ ]:
# résumé global d'execution
import json
summary_file = Path(morlet_cfg.output_dir) / "run_summary.json"
with open(summary_file, "r") as f:
    summary = json.load(f)
summary

{'n_sessions': 2,
 'n_errors': 0,
 'errors': [],
 'config': {'root_dir': '/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog',
  'output_dir': '/home/aube/Documents/article_neuronal_stimic/effets_cog//results_morlet_exploratoire',
  'pre_length': 3.0,
  'post_length': 3.0,
  'epsilon': 0.1,
  'allow_global_baseline': True,
  'do_notch': True,
  'notch_freqs': [50.0, 100.0, 150.0],
  'notch_q': 30.0,
  'do_highpass': True,
  'highpass_hz': 0.1,
  'fmin': 4.0,
  'fmax': 150.0,
  'n_freqs': 20,
  'freq_scale': 'linear',
  'n_cycles': 7.0,
  'decim': 16,
  'n_jobs': 1,
  'baseline_mode': 'trial_pre',
  'baseline_stat': 'median',
  'metrics_to_compute': ['logratio', 'percent', 'zscore', 'subtract'],
  'eps': 1e-12,
  'save_raw_epochs': False,
  'save_filtered_signal_preview': False,
  'make_figures': True,
  'figure_dpi': 150,
  'max_cols_per_figure': 3,
  'cmap_raw': 'viridis',
  'cmap_metric_div': 'RdBu_r',
  'raw_display_mode': 'log10',
  'z_threshold': 3.0,
  'linep

In [ ]:
# Variante de Morlet sans figures, et sans verbose, pour accélérer le calcul et creation des .npy
morlet_cfg_fast = MorletConfig(
    **{**morlet_cfg.__dict__, "make_figures": False, "verbose": False}
)

In [ ]:
# Variante de Morlet avec baseline globale au lieu de baseline pre-stim
morlet_cfg_global_base = MorletConfig(
    **{
        **morlet_cfg.__dict__,
        "baseline_mode": "global_pre_first_stim",
        "allow_global_baseline": True,
    }
)

In [ ]:
# Variante de Morlet avec exploration fréquentielle plus importante
morlet_cfg_dense = MorletConfig(
    **{
        **morlet_cfg.__dict__,
        "n_freqs": 40,
        "freq_scale": "semilog",
    }
)